In [ ]:
# 1. Install required packages (100% Offline Mode)
import os
import sys
import subprocess
import glob

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

offline_pkg_dir = "/kaggle/input/datasets/mduy2911/offline-packages"
print(f"Installing offline packages from: {offline_pkg_dir}...")
wheels = glob.glob(os.path.join(offline_pkg_dir, "*.whl"))

if wheels:
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"] + wheels
    subprocess.run(cmd, check=True)
    print("Offline installation completed successfully!")
else:
    raise FileNotFoundError(f"No offline wheels found in {offline_pkg_dir}!")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# 2. TRAINING HYPERPARAMETERS (Optimized for RTX Pro 6000 Ada / 96GB VRAM)
import os

MODEL_ID = "/kaggle/input/datasets/mduy2911/model-cache"

# --- RTX Pro 6000 Hardware Optimizations ---
USE_QLORA = False
GRADIENT_CHECKPOINTING = True

# --- Training Hyperparameters ---
MAX_LENGTH = 4096            # Maximum sequence length
BATCH_SIZE = 16           # Physical batch size optimized for RTX 6000
GRADIENT_ACCUMULATION = 2  # Gradient accumulation steps (Effective batch size = 32)
EPOCHS = 2                  # Number of training epochs
LEARNING_RATE = 1e-4        # Learning rate
OUTPUT_DIR = "/kaggle/working/results_physics"  # Output directory on Kaggle

os.environ["WANDB_MODE"] = "disabled"
USE_WANDB = False


In [ ]:
# 3. Load Physics Dataset and Physics Prompt
import os
import json

physics_path = "/kaggle/input/datasets/mduy2911/folc-train/physics_distillation.json"

def load_physics_dataset(path):
    samples = []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        inp = item.get("input", "")
        out = item.get("output", "")
        if inp and out:
            samples.append({"input": inp, "output": out})
    print(f"Loaded {len(samples)} physics samples from {os.path.basename(path)}")
    return samples

physics_samples = load_physics_dataset(physics_path)

physics_system_prompt = """You are a precise physics reasoning agent. Solve the provided physics problem by analyzing its states, setting up symbolic equations, and writing executable SymPy code.

<REASONING_POLICY_OVERRIDE>
A <reasoning_policies> block may be provided alongside the problem.

If present:
1. Treat it as the primary reasoning guidance.
2. Follow its topology rules, state representations, coordinate conventions, and solution procedures.
3. Apply the underlying reasoning pattern — do not copy example expressions verbatim.
4. Override a policy only when required for physical validity or SymPy executability.
</REASONING_POLICY_OVERRIDE>

<OPERATING_CONSTRAINTS>

OUTPUT FORMAT:
Return ONLY a single valid JSON object. Do not wrap in markdown. Do not include any text before or after the JSON.

{
  "thought": "...",
  "physics_analysis": ["..."],
  "algebraic_reasoning": ["..."],
  "python_code": "...",
  "json_terminated": true
}

──────────────────────────────

THOUGHT

Format: <detected structure>. <activated reasoning pattern>. <solution strategy>.

Requirements:
- Keep reasoning concise and high-level.
- Do not perform calculations.
- Do not reveal numerical results.

──────────────────────────────

PHYSICS_ANALYSIS

Describe:
- Physical states
- Topology and governing constraints
- Target quantities

Requirements:
- Do not perform calculations.
- Do not derive equations.
- Do not reveal final numerical values.

──────────────────────────────

ALGEBRAIC_REASONING

Describe:
- Setup
- Transformation
- Solve
- Extraction

Requirements:
- Do not perform calculations.
- Do not reveal final numerical values.

──────────────────────────────

STATE SEPARATION

Treat distinct physical states independently.
Do not combine equations from mutually exclusive states.
Use explicit state suffixes when states change.

Preferred suffixes: _init, _new, _res, _final

──────────────────────────────

SYMPY CODE REQUIREMENTS

Always begin with:
import sympy as sp

Numerical constants:
- Use sp.Float('1.23') or sp.Float('1e-6')
- Never use raw Python floats or ints in equations
- Use sp.Float('4') * sp.pi, not sp.Float('4*pi')

Variable rules:
- Define all symbols explicitly before use
- No undefined variables

Solving rules:
- Always solve symbolically with sp.solve() before evaluating numerically
- Never call float() on sp.Eq(...), relational objects, or unsolved equations
- Prefer physically meaningful real (positive) roots

Numerical evaluation:
- When computing vector norms, distances, or square roots, force immediate evaluation:
  use float(expr.evalf()) to prevent symbolic graph explosion

Code format:
- Write as a single continuous flat string
- Separate all statements with '; ' (semicolon + space)
- Never use \n, def, loops, or conditional branches
- Write pure sequential, line-by-line imperative calculations

Final variables (always last, always defined):
ans = [value] or [value1, value2]
unit = ['SI_unit'] or ['unit1', 'unit2']

──────────────────────────────

SI UNIT POLICY

Assume all inputs are pre-converted to SI units.
Output must use SI base or derived units only.
Never use engineering prefixes (mA, μF, kΩ, etc.).
Use raw SI magnitudes: A, V, F, H, ohm, m, kg, s, N, J, W, C, T

──────────────────────────────

ANSWER FORMAT

Numerical:
  ans = [value]
  unit = ['SI_unit']

Multiple numerical:
  ans = [value1, value2]
  unit = ['unit1', 'unit2']

Text:
  ans = ['description']
  unit = ['-']

Formula:
  ans = ['formula']
  unit = ['-']

Requirements:
- ans must contain only scalars, strings, or flat lists
- Never return dicts or matrices inside ans
- ans and unit must have matching lengths
- Output raw full-precision floats — never use round() or string formatting on numeric values

──────────────────────────────

FINAL VALIDATION

Before responding, ensure:
1. Response is valid JSON
2. All required fields exist
3. python_code is fully executable
4. All variables are defined
5. ans is produced
6. unit is produced
7. SI units are used throughout
8. json_terminated is exactly true

Return the JSON object only.

</OPERATING_CONSTRAINTS>""".strip()


In [ ]:
# 4. Initialize Tokenizer and Load Helper Functions
import os
import sys
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Check hardware bfloat16 support
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    use_bf16 = True
    use_fp16 = False
    print("GPU supports bfloat16. Using bfloat16 compute (Optimal for Ampere/Ada/Hopper GPUs like RTX Pro 6000).")
else:
    compute_dtype = torch.float16
    use_bf16 = False
    use_fp16 = True
    print("Using float16 compute (Standard for Turing/Pascal/Volta GPUs or CPU).")

# Select the most optimal Attention implementation
attn_impl = "sdpa"
try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print("FlashAttention-2 is installed. Using flash_attention_2.")
except ImportError:
    print("FlashAttention-2 not found. Using PyTorch Native SDPA (Scaled Dot Product Attention).")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True, 
    local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base_model():
    print("Loading a fresh instance of the base model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=compute_dtype if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
        attn_implementation=attn_impl,
        local_files_only=True
    )
    model.config.use_cache = False
    return model

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("GPU and system memory cleaned.")

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Optimized LoRA Configuration (Rank = 32, Alpha = 64) for logical capacity
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=target_modules,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
# 5. Format Dataset (Chat Template) and split Train/Val
import os
import json
import random
from datasets import Dataset

# --- Split Physics dataset deterministically ---
random.Random(42).shuffle(physics_samples)
split_idx_phys = int(len(physics_samples) * 0.9)
train_physics = physics_samples[:split_idx_phys]
val_physics = physics_samples[split_idx_phys:]

# --- Format training and validation samples ---
def format_samples(physics_list):
    formatted = []
    
    # Format Physics samples
    for item in physics_list:
        physics_input = item["input"]
        physics_output = item["output"]
        
        formatted.append({
            "messages": [
                {"role": "system", "content": physics_system_prompt},
                {"role": "user", "content": physics_input.strip()},
                {"role": "assistant", "content": physics_output.strip()}
            ]
        })
        
    return formatted

formatted_train = format_samples(train_physics)
formatted_val = format_samples(val_physics)

train_dataset = Dataset.from_list(formatted_train)
val_dataset = Dataset.from_list(formatted_val)

def apply_template(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

train_dataset = train_dataset.map(apply_template, batched=True, remove_columns=["messages"])
val_dataset = val_dataset.map(apply_template, batched=True, remove_columns=["messages"])

# Shuffle the final HF datasets
train_dataset = train_dataset.shuffle(seed=42)
val_dataset = val_dataset.shuffle(seed=42)

print(f"Physics Train/Val Split: train={len(train_physics)}, val={len(val_physics)}")
print(f"Total Train size: {len(train_dataset)}, Total Val size: {len(val_dataset)}")


In [ ]:
# 6. Configure SFTTrainer and start training
import os
import glob
import torch
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, DataCollatorForLanguageModeling
from typing import Dict, Union, Any, Optional, List, Tuple

class LossLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                print(f"Step {state.global_step}: training_loss = {logs['loss']:.6f}")
            if "eval_loss" in logs:
                print(f"Step {state.global_step}: validation_loss = {logs['eval_loss']:.6f}")

class CustomDataCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer=tokenizer, mlm=False, *args, **kwargs)
        self.response_template = response_template
        self.response_token_ids = self.tokenizer.encode(self.response_template, add_special_tokens=False)
        
    def __call__(self, examples):
        batch = super().__call__(examples)
        labels = batch["labels"]
        for i in range(labels.size(0)):
            input_ids = batch["input_ids"][i].tolist()
            response_idx = -1
            n_template = len(self.response_token_ids)
            for j in range(len(input_ids) - n_template + 1):
                if input_ids[j:j+n_template] == self.response_token_ids:
                    response_idx = j + n_template
                    break
            
            if response_idx != -1:
                labels[i, :response_idx] = -100
                
        return batch

def train_model(train_dataset, val_dataset, output_dir):
    clean_memory()
    
    # Mathematically exact, dynamic warmup steps calculation based on actual dataset size
    num_samples = len(train_dataset)
    effective_batch_size = BATCH_SIZE * GRADIENT_ACCUMULATION
    steps_per_epoch = num_samples // effective_batch_size
    if num_samples % effective_batch_size != 0:
        steps_per_epoch += 1
    total_steps = steps_per_epoch * EPOCHS
    
    # Target exactly 5% warmup steps of total training steps (HF v5.2 compliant integer steps)
    warmup_steps = max(1, int(total_steps * 0.05))
    print(f"Training on {num_samples} samples. Steps per epoch: {steps_per_epoch}. Total steps: {total_steps}. Dynamic warmup steps: {warmup_steps}")
    
    base_model = load_base_model()
    
    # Resume from previous checkpoints if adapter_config.json exists in output_dir
    adapter_config_path = os.path.join(output_dir, "adapter_config.json")
    if os.path.exists(adapter_config_path):
        print(f"Resuming PEFT adapter weights from {output_dir}...")
        model = PeftModel.from_pretrained(base_model, output_dir, is_trainable=True)
    else:
        print("Initializing a new PEFT adapter...")
        model = get_peft_model(base_model, peft_config)
        
    model.print_trainable_parameters()
    
    # Highly Optimized SFTConfig with Cosine Decay Scheduler & Warmup
    sft_config = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",             # Cosine learning rate decay scheduler
        warmup_steps=warmup_steps,              # Compliant, dynamic warm-up steps to stabilize updates
        fp16=use_fp16,
        bf16=use_bf16,
        logging_steps=1,
        logging_first_step=True,
        optim="adamw_torch",
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        dataset_text_field="text",
        max_length=MAX_LENGTH,
        neftune_noise_alpha=None,
        dataloader_num_workers=2,
        dataloader_pin_memory=True
    )
    
    response_template = "<|im_start|>assistant\n"
    collator = CustomDataCollator(
        response_template=response_template, 
        tokenizer=tokenizer
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=sft_config,
        data_collator=collator,
        callbacks=[LossLoggingCallback()]
    )
    
    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    resume_path = None
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        resume_path = checkpoints[-1]
        print(f"Found checkpoints. Resuming training from: {resume_path}")
        
    trainer.train(resume_from_checkpoint=resume_path)
    
    print(f"Saving best validation adapter weights and tokenizer to {output_dir}...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print("Training completed successfully!")
    
    # Clean up model & trainer from memory
    del trainer
    del model
    del base_model
    clean_memory()

# Run training
train_model(train_dataset, val_dataset, OUTPUT_DIR)
